# Wage efficiency in European football — 01 · Data collection

Which clubs get the most (and least) out of their wage bill? This notebook builds the club-season dataset:
results, wages and squad market value for the Big 5 leagues × 5 seasons.

**Pilot first:** LaLiga 2025/26. Once every source loads and matches cleanly, scale to all leagues and seasons (last section).

| Source | Data | Notes |
|---|---|---|
| football-data.co.uk | Match results → points, PPG, GD, position | Free CSVs |
| FBref (Capology) | Squad annual wages, % estimated | FBref blocks Colab, so pages are saved from the browser into Drive |
| Transfermarkt (dcaribou dataset, Kaggle) | Squad market value at season start | Weekly-updated dump |

In [ ]:
!pip -q install rapidfuzz kagglehub

from google.colab import drive
drive.mount('/content/drive')

import io, re, time
from pathlib import Path
import numpy as np
import pandas as pd
import requests
from rapidfuzz import process, fuzz

HEADERS = {"User-Agent": "Mozilla/5.0 (Windows NT 10.0; Win64; x64) AppleWebKit/537.36 (KHTML, like Gecko) Chrome/124.0 Safari/537.36"}

def fd_code(season):
    # '2025-2026' -> '2526'
    a, b = season.split("-")
    return a[2:] + b[2:]

def season_end(season):
    return int(season.split("-")[1])
import matplotlib.pyplot as plt

BASE = Path('/content/drive/MyDrive/proyecto_eficiencia')
RAW = BASE / 'data' / 'raw'
PROCESSED = BASE / 'data' / 'processed'
for p in (RAW, PROCESSED):
    p.mkdir(parents=True, exist_ok=True)

In [ ]:
# per-league ids for every source (Championship dropped: FBref has no wage data for it)
LEAGUES = {
    'laliga':       {'fd': 'SP1', 'fbref_id': 12, 'fbref_name': 'La-Liga',        'tm': 'ES1'},
    'premier':      {'fd': 'E0',  'fbref_id': 9,  'fbref_name': 'Premier-League', 'tm': 'GB1'},
    'seriea':       {'fd': 'I1',  'fbref_id': 11, 'fbref_name': 'Serie-A',        'tm': 'IT1'},
    'bundesliga':   {'fd': 'D1',  'fbref_id': 20, 'fbref_name': 'Bundesliga',     'tm': 'L1'},
    'ligue1':       {'fd': 'F1',  'fbref_id': 13, 'fbref_name': 'Ligue-1',        'tm': 'FR1'},
}
SEASONS = ['2021-2022', '2022-2023', '2023-2024', '2024-2025', '2025-2026']

LEAGUE, SEASON = 'laliga', '2025-2026'  # pilot

## 1. Results → league table

Table rebuilt from match results: points, PPG, goal difference and position.
Point deductions are ignored on purpose (we measure sporting output), and ties are broken by GD then GF (no head-to-head).

In [ ]:
def load_results(league, season, raw_dir):
    path = raw_dir / "results" / f"{league}_{season}.csv"
    if path.exists():
        return pd.read_csv(path)
    code = LEAGUES[league]["fd"]
    url = f"https://www.football-data.co.uk/mmz4281/{fd_code(season)}/{code}.csv"
    df = pd.read_csv(url, encoding="latin-1")
    df = df[["Date", "HomeTeam", "AwayTeam", "FTHG", "FTAG"]].dropna()
    path.parent.mkdir(parents=True, exist_ok=True)
    df.to_csv(path, index=False)
    return df

def build_table(results):
    # one row per team per match, then aggregate
    home = pd.DataFrame({"team": results["HomeTeam"], "gf": results["FTHG"], "ga": results["FTAG"]})
    away = pd.DataFrame({"team": results["AwayTeam"], "gf": results["FTAG"], "ga": results["FTHG"]})
    long = pd.concat([home, away], ignore_index=True)
    long["pts"] = np.select([long.gf > long.ga, long.gf == long.ga], [3, 1], 0)
    t = long.groupby("team").agg(played=("pts", "size"), points=("pts", "sum"),
                                 gf=("gf", "sum"), ga=("ga", "sum")).reset_index()
    t["gd"] = t.gf - t.ga
    t["ppg"] = t.points / t.played
    # tiebreak approx: pts > gd > gf (no h2h, no point deductions)
    t = t.sort_values(["points", "gd", "gf"], ascending=False).reset_index(drop=True)
    t["position"] = t.index + 1
    return t


results = load_results(LEAGUE, SEASON, RAW)
table = build_table(results)
table.head(20)

## 2. Wages (Capology via FBref)

Squad-level annual wages in €, plus the share of the bill that Capology estimates (useful as a data-quality flag).
FBref blocks requests from Colab (403), so each page is saved once from the browser and read from Drive:

1. Run the checklist cell below: it lists every missing page with its URL and the exact file name.
2. Open the URL, Ctrl+S → *Webpage, HTML only*, rename the file to the name shown.
3. Upload it to `proyecto_eficiencia/data/raw/fbref_html/` and re-run.

For the pilot only `laliga_2025-2026_wages.html` is needed.

In [ ]:
FBREF_DIR = RAW / 'fbref_html'
FBREF_DIR.mkdir(parents=True, exist_ok=True)

def fbref_url(league, season):
    cfg = LEAGUES[league]
    return f"https://fbref.com/en/comps/{cfg['fbref_id']}/{season}/wages/{season}-{cfg['fbref_name']}-Wages"

def fbref_checklist(pairs):
    missing = [(f'{lg}_{ss}_wages.html', fbref_url(lg, ss)) for lg, ss in pairs
               if not (FBREF_DIR / f'{lg}_{ss}_wages.html').exists()]
    print(f'{len(missing)} pages missing')
    for name, url in missing:
        print(f'{name}  ->  {url}')

fbref_checklist([(LEAGUE, SEASON)])  # pilot; full list in the scale-up section

In [ ]:
def get_html(url, cache_path, sleep=6):
    cache_path.parent.mkdir(parents=True, exist_ok=True)
    if cache_path.exists():
        return cache_path.read_text(encoding="utf-8", errors="ignore")
    r = requests.get(url, headers=HEADERS, timeout=30)
    if r.status_code != 200:
        # fbref blocks cloud IPs (403): save the page from your browser instead
        raise RuntimeError(f"{r.status_code} on {url} -> save the page manually as {cache_path}")
    cache_path.write_text(r.text, encoding="utf-8")
    time.sleep(sleep)  # fbref rate limit
    return r.text

def parse_eur(value):
    # '€ 282,650,000 (£ ..., $...)' or '£ 3,640,000 (€ 4,312,479, $...)'
    m = re.search(r"€\s*([\d,]+)", str(value))
    return float(m.group(1).replace(",", "")) if m else np.nan

def parse_squad_wages(html):
    html = html.replace("<!--", "").replace("-->", "")  # fbref hides tables in comments
    tables = pd.read_html(io.StringIO(html))
    for t in tables:
        if isinstance(t.columns, pd.MultiIndex):
            t.columns = t.columns.get_level_values(-1)
        cols = set(map(str, t.columns))
        if {"Squad", "Annual Wages"} <= cols and "Player" not in cols:
            out = t[t["Squad"].notna() & (t["Squad"] != "Squad")].copy()
            out["annual_wages_eur"] = out["Annual Wages"].map(parse_eur)
            est = out["% Estimated"] if "% Estimated" in out else np.nan
            out["pct_estimated"] = pd.to_numeric(pd.Series(est, index=out.index).astype(str).str.rstrip("%"), errors="coerce")
            return out[["Squad", "annual_wages_eur", "pct_estimated"]].rename(columns={"Squad": "team"}).reset_index(drop=True)
    raise ValueError("squad wages table not found")

def load_wages(league, season, raw_dir):
    cfg = LEAGUES[league]
    url = f"https://fbref.com/en/comps/{cfg['fbref_id']}/{season}/wages/{season}-{cfg['fbref_name']}-Wages"
    html = get_html(url, raw_dir / "fbref_html" / f"{league}_{season}_wages.html")
    return parse_squad_wages(html)


wages = load_wages(LEAGUE, SEASON, RAW)
wages.sort_values('annual_wages_eur', ascending=False)

## 3. Squad market value (Transfermarkt)

Latest valuation of each player before 1 September (season start), summed by club.
If `kagglehub` asks for credentials: Kaggle → Settings → *Create New Token*, upload `kaggle.json` and re-run;
or download the dataset zip manually and unzip it into `data/raw/transfermarkt/`.

In [ ]:
import kagglehub

tm_dir = RAW / 'transfermarkt'
if not (tm_dir / 'player_valuations.csv').exists():
    tm_dir = Path(kagglehub.dataset_download('davidcariboo/player-scores'))

valuations = pd.read_csv(tm_dir / 'player_valuations.csv')
clubs = pd.read_csv(tm_dir / 'clubs.csv')
valuations.columns.tolist()  # check the competition column is there

In [ ]:
def squad_value(valuations, clubs, league, season):
    # latest valuation per player before season start, summed by the club he was at
    cutoff = pd.Timestamp(f"{season.split('-')[0]}-09-01")
    tm_code = LEAGUES[league]["tm"]
    v = valuations.copy()
    v["date"] = pd.to_datetime(v["date"])
    v = v[v.date <= cutoff].sort_values("date").groupby("player_id").tail(1)
    v = v[v.date >= cutoff - pd.Timedelta(days=365)]  # drop stale/retired players
    if "player_club_domestic_competition_id" in v:
        v = v[v.player_club_domestic_competition_id == tm_code]
    out = v.groupby("current_club_id")["market_value_in_eur"].sum().reset_index()
    out = out.merge(clubs[["club_id", "name"]], left_on="current_club_id", right_on="club_id")
    if "player_club_domestic_competition_id" not in v:
        out = out.merge(clubs[["club_id", "domestic_competition_id"]], on="club_id")
        out = out[out.domestic_competition_id == tm_code]
    return out[["name", "market_value_in_eur"]].rename(columns={"name": "team", "market_value_in_eur": "squad_value_eur"})


values = squad_value(valuations, clubs, LEAGUE, SEASON)
values.sort_values('squad_value_eur', ascending=False)

## 4. Entity matching

Each source spells clubs differently ("Ath Madrid", "Atlético Madrid", "Club Atlético de Madrid").
football-data names are the master key. Fuzzy matching suggests a match for every other source; rows flagged `needs_review`
(low score or duplicated match) must be checked by hand in `club_mapping.csv`. Reviewed rows are kept between runs.

In [ ]:
def suggest_mapping(base_names, other_names, source, threshold=85):
    # fuzzy suggestions, to be reviewed by hand
    rows = []
    for b in base_names:
        match, score, _ = process.extractOne(b, list(other_names), scorer=fuzz.token_set_ratio)
        rows.append({"team": b, "source": source, "source_name": match, "score": score,
                     "needs_review": score < threshold})
    return pd.DataFrame(rows)

def apply_mapping(df, mapping, source):
    m = mapping[mapping.source == source][["team", "source_name"]]
    return df.rename(columns={"team": "source_name"}).merge(m, on="source_name", how="inner").drop(columns="source_name")


MAPPING_PATH = BASE / 'data' / 'club_mapping.csv'

sources = {'fbref': wages.team, 'tm': values.team}
suggested = pd.concat([suggest_mapping(table.team, names, src) for src, names in sources.items()], ignore_index=True)

# same source name matched to two clubs = wrong somewhere
dup = suggested.duplicated(['source', 'source_name'], keep=False)
suggested.loc[dup, 'needs_review'] = True

if MAPPING_PATH.exists():
    existing = pd.read_csv(MAPPING_PATH)
    new = suggested.merge(existing[['team', 'source']], on=['team', 'source'], how='left', indicator=True)
    new = new[new['_merge'] == 'left_only'].drop(columns='_merge')
    mapping = pd.concat([existing, new], ignore_index=True)
else:
    mapping = suggested

mapping.to_csv(MAPPING_PATH, index=False)
mapping[mapping.needs_review]

**Manual step:** open `club_mapping.csv` (Drive → Google Sheets), fix the wrong `source_name` values in the flagged rows,
set `needs_review` to `False`, save as CSV and re-run the next cell.

## 5. Club-season table + quality checks

In [ ]:
mapping = pd.read_csv(MAPPING_PATH)
pending = mapping[mapping.needs_review.astype(str).str.lower() == 'true']
if len(pending):
    print(f'{len(pending)} mappings still need review')

club_season = (table
    .merge(apply_mapping(wages, mapping, 'fbref'), on='team', how='left')
    .merge(apply_mapping(values, mapping, 'tm'), on='team', how='left')
    .assign(league=LEAGUE, season=SEASON))

# checks: one row per club, no gaps, sane magnitudes
print('clubs:', len(club_season), '| duplicated:', club_season.team.duplicated().sum())
print(club_season[['annual_wages_eur', 'squad_value_eur']].isna().sum())
print('wage bill range (€m):', (club_season.annual_wages_eur.agg(['min', 'max']) / 1e6).round(1).tolist())
print('mean % estimated:', round(club_season.pct_estimated.mean(), 1))

club_season.to_csv(PROCESSED / f'club_season_{LEAGUE}_{SEASON}.csv', index=False)
club_season.sort_values('position')

## 6. Sanity check

Before modelling: does wage spend line up with PPG at all? Log scale on wages, since returns diminish at the top.

In [ ]:
fig, ax = plt.subplots(figsize=(9, 6))
d = club_season.dropna(subset=['annual_wages_eur'])
ax.scatter(d.annual_wages_eur / 1e6, d.ppg)
for _, r in d.iterrows():
    ax.annotate(r.team, (r.annual_wages_eur / 1e6, r.ppg), fontsize=8, xytext=(4, 2), textcoords='offset points')
ax.set_xscale('log')
ax.set_xlabel('Annual wage bill (€m, log scale)')
ax.set_ylabel('Points per game')
ax.set_title(f'{LEAGUE} {SEASON}: wages vs points per game')
plt.show()

print('corr log(wages) vs PPG:', round(np.corrcoef(np.log(d.annual_wages_eur), d.ppg)[0, 1], 2))

## 7. Scale up (run once the pilot is clean)

Loops over every league and season. All 25 FBref pages are already in Drive (checklist below lists the missing ones);
seasons with a missing page are skipped and reported.
Review `club_mapping.csv` again after the first full run — promoted/relegated clubs will add new names.

In [ ]:
def collect(league, season):
    t = build_table(load_results(league, season, RAW))
    w = load_wages(league, season, RAW)
    v = squad_value(valuations, clubs, league, season)
    return t, {'fbref': w, 'tm': v}

fbref_checklist([(lg, ss) for lg in LEAGUES for ss in SEASONS])

RUN_ALL = True

if RUN_ALL:
    frames, suggestions = [], []
    for lg in LEAGUES:
        for ss in SEASONS:
            try:
                t, srcs = collect(lg, ss)
            except Exception as err:
                print(f'skip {lg} {ss}: {err}')
                continue
            suggestions += [suggest_mapping(t.team, s.team, name) for name, s in srcs.items()]
            frames.append((lg, ss, t, srcs))
            print(f'ok {lg} {ss}: {len(t)} clubs')

    sugg = pd.concat(suggestions, ignore_index=True).drop_duplicates(['team', 'source', 'source_name'])
    existing = pd.read_csv(MAPPING_PATH)
    new = sugg.merge(existing[['team', 'source']], on=['team', 'source'], how='left', indicator=True)
    new = new[new['_merge'] == 'left_only'].drop(columns='_merge')
    pd.concat([existing, new], ignore_index=True).to_csv(MAPPING_PATH, index=False)
    print(f'{len(new)} new mappings added -> review needs_review rows, then build the panel')

### Manual mapping fixes (reviewed after the first full run)

Fuzzy matching got these wrong: "Man United" also matched Leeds/Sheffield United (shared word *United*), Bielefeld matched Leverkusen
(FBref calls it *Arminia*), and on Transfermarkt Man City, Man United, Rennes and St Etienne matched other clubs because TM uses long official names.
Fixes are applied by exact name (FBref) or by keyword search over TM club names, then printed for a quick check.

In [ ]:
import unicodedata

def norm(s):
    return unicodedata.normalize('NFKD', str(s)).encode('ascii', 'ignore').decode().lower()

FBREF_FIXES = {'Man United': 'Manchester Utd', 'Bielefeld': 'Arminia'}
TM_FIXES = {'Man City': 'manchester city', 'Man United': 'manchester united', 'Rennes': 'rennais', 'St Etienne': 'saint-etienne'}

if RUN_ALL:
    mapping = pd.read_csv(MAPPING_PATH)
    drop = ((mapping.team.isin(FBREF_FIXES) & (mapping.source == 'fbref')) |
            (mapping.team.isin(TM_FIXES) & (mapping.source == 'tm')))
    mapping = mapping[~drop]

    rows = [{'team': t, 'source': 'fbref', 'source_name': n} for t, n in FBREF_FIXES.items()]
    tm_names = pd.concat([srcs['tm'].team for *_, srcs in frames]).unique()
    for t, key in TM_FIXES.items():
        hits = [n for n in tm_names if key in norm(n)]
        print(f'{t:12s} -> {hits}')
        rows += [{'team': t, 'source': 'tm', 'source_name': h} for h in hits[:1]]

    fixes = pd.DataFrame(rows).assign(score=100.0)
    mapping = pd.concat([mapping, fixes], ignore_index=True).assign(needs_review=False)
    mapping.to_csv(MAPPING_PATH, index=False)

In [ ]:
def build_panel():
    mapping = pd.read_csv(MAPPING_PATH)
    rows = []
    for lg, ss, t, srcs in frames:
        cs = t.copy()
        for name, s in srcs.items():
            cs = cs.merge(apply_mapping(s, mapping, name), on='team', how='left')
        rows.append(cs.assign(league=lg, season=ss))
    panel = pd.concat(rows, ignore_index=True)
    dups = panel.duplicated(['league', 'season', 'team']).sum()
    assert dups == 0, f'{dups} duplicated club-seasons: a club is mapped to two source names'
    panel.to_csv(PROCESSED / 'club_season_panel.csv', index=False)
    return panel

if RUN_ALL:
    panel = build_panel()
    print(panel.groupby(['league', 'season']).size().unstack())
    print(panel[['annual_wages_eur', 'squad_value_eur']].isna().mean().round(3))